# MonoSplat — Production Gaussian Splatting Notebook
**Version 3.0 — Stable | Deterministic | VRAM-Aware | Research-Grade**

---

## Design Philosophy

This notebook is built around one non-negotiable principle:

> **A stable 80% quality result is vastly more valuable than a crashing 100% quality attempt.**

Every design decision — conservative defaults, hard Gaussian caps, pre-densification checkpoints,
VRAM guards, NaN detection — flows from that philosophy.

### What this notebook provides

- **Deterministic training**: fixed seeds, reproducible runs, no silent state drift
- **VRAM-aware loop**: every iteration tracks allocated/reserved memory; hard brakes before OOM
- **Resumable checkpoints**: crash mid-run? Resume from the last saved iteration automatically
- **Drive persistence**: all checkpoints, previews, exports, and logs sync to Google Drive
- **Hardware presets**: T4 (conservative), L4 (balanced), A100 (high-quality) — auto-detected
- **Crash protection**: CUDA OOM, NaN explosions, Drive disconnects — all handled with diagnostics
- **Benchmark mode**: known-scene benchmarks to validate your setup before training your own data

### Hardware support

| GPU | VRAM | Preset | Max Gaussians | Resolution |
|-----|------|--------|---------------|------------|
| T4 (free Colab) | 15 GB | conservative | 100,000 | 640×360 |
| L4 (Colab Pro) | 22 GB | balanced | 200,000 | 960×540 |
| A100 (Colab Pro+) | 40 GB | high-quality | 500,000 | 1280×720 |

### Prerequisites

1. ✅ Complete COLMAP reconstruction on your desktop (FFmpeg → COLMAP → `sparse_text/`)
2. ✅ Package your job: `python scripts/zip_for_colab.py <job_id>`
3. ✅ Runtime → Change runtime type → **T4 GPU** → Save
4. ✅ Run cells **top to bottom** — do not skip any cell

### Cell execution order

```
 [1] GPU Verification          ← MUST see GPU here
 [2] Environment Setup         ← Installs dependencies
 [3] Google Drive Mount        ← Enables crash-safe persistence
 [4] Hardware Preset           ← Auto-selects T4/L4/A100 settings
 [5] Dataset Upload            ← Upload monosplat_job.zip
 [6] Dataset Validation        ← Hard-fails on bad data
 [7] Gaussian Initialization   ← Safe, clamped, validated
 [8] Training Loop             ← VRAM-aware, resumable, checkpointed
 [9] Preview Rendering         ← Save renders to Drive
[10] Metrics & Logging         ← Structured JSON logs
[11] Export                    ← .splat, .ply, logs bundle
[12] Debug Utilities           ← For diagnosing failures
```


---
## Cell 1 — GPU Verification

**This cell must succeed before anything else.** It checks that:
- A CUDA GPU is available
- VRAM is sufficient for training
- Memory allocator is configured for stability
- Python version is compatible

If this fails: `Runtime → Change runtime type → T4 GPU → Save → Reconnect`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 1 — GPU VERIFICATION
#  Purpose : Confirm CUDA GPU is present and VRAM is sufficient.
#  Aborts  : If no GPU, or VRAM < 8 GB.
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

# ── 1a. Configure memory allocator BEFORE torch import ──────────────────────
# expandable_segments reduces fragmentation that causes OOM mid-training.
# This MUST be set before torch is imported.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"]     = "0"   # async is faster; set to 1 to debug crashes

# ── 1b. nvidia-smi sanity check ─────────────────────────────────────────────
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    print("❌  No GPU detected!")
    print("   → Runtime → Change runtime type → T4 GPU → Save → Reconnect")
    raise SystemExit("ABORT: No GPU. Change runtime type and restart.")

print("GPU hardware report:")
print("─" * 60)
for line in smi.stdout.split("\n")[:13]:
    print(line)
print("─" * 60)

# ── 1c. Python version ───────────────────────────────────────────────────────
assert sys.version_info >= (3, 9), f"Python 3.9+ required, got {sys.version}"
print(f"\n✅  Python {sys.version.split()[0]}")

# ── 1d. Torch + CUDA validation ──────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    print("❌  torch.cuda.is_available() returned False.")
    print("   → This means PyTorch was installed without CUDA support.")
    print("   → Try: Runtime → Factory reset runtime → re-run from Cell 1.")
    raise SystemExit("ABORT: CUDA not available.")

props    = torch.cuda.get_device_properties(0)
VRAM_GB  = props.total_memory / 1e9
GPU_NAME = props.name

print(f"✅  PyTorch  : {torch.__version__}")
print(f"✅  CUDA     : {torch.version.cuda}")
print(f"✅  cuDNN    : {torch.backends.cudnn.version()}")
print(f"✅  GPU      : {GPU_NAME}")
print(f"✅  VRAM     : {VRAM_GB:.1f} GB")

# ── 1e. VRAM floor check ─────────────────────────────────────────────────────
# Training is not practical below 8 GB. Warn at 12 GB, error below 8 GB.
if VRAM_GB < 8:
    print(f"\n❌  Only {VRAM_GB:.1f} GB VRAM — minimum 8 GB required for stable training.")
    raise SystemExit("ABORT: Insufficient VRAM.")
elif VRAM_GB < 12:
    print(f"\n⚠️  {VRAM_GB:.1f} GB VRAM — marginal. Training may be slow and Gaussian cap will be very low.")
else:
    print(f"\n✅  VRAM sufficient for stable training.")

# ── 1f. Clear residual VRAM from previous runs ───────────────────────────────
torch.cuda.empty_cache()
import gc; gc.collect()

free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"✅  Free VRAM after clear: {free_gb:.1f} GB")
print(f"✅  Memory allocator: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")
print("\n→  Run Cell 2 to install dependencies.")


---
## Cell 2 — Environment Setup

Installs all required packages with pinned versions for reproducibility.  
Validates gsplat CUDA rasterizer — without it, training is ~20× slower.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 2 — ENVIRONMENT SETUP
#  Purpose : Install and validate all training dependencies.
#  Runtime : ~60 seconds on first run.
#  Why pinned? Unpinned installs can silently change behavior across Colab
#  sessions, making failures impossible to reproduce or debug.
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, importlib, time

def pip_install(*packages, quiet=True):
    """Install packages and return True on success."""
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd.extend(packages)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌  pip install failed for {packages}")
        print(result.stderr[-2000:])
        return False
    return True

# ── 2a. Core dependencies ────────────────────────────────────────────────────
# These are required for data loading, image processing, and I/O.
# Versions pinned to what was tested stable on Colab + PyTorch 2.x
print("Installing core dependencies...")
t0 = time.time()

core_packages = [
    "pillow>=10.0.0",
    "pyyaml>=6.0",
    "plyfile>=0.9",
    "tqdm>=4.66",
    "numpy>=1.24",
    "scipy>=1.11",        # used for KD-tree in point cloud operations
    "imageio>=2.31",      # preview frame writing
]

if not pip_install(*core_packages):
    raise RuntimeError("Core dependency installation failed. Try: Runtime → Factory reset runtime.")

print(f"  Core deps installed in {time.time()-t0:.0f}s")

# ── 2b. gsplat CUDA rasterizer ───────────────────────────────────────────────
# gsplat provides the CUDA rasterization backend that makes training tractable.
# Without it, the Python fallback renderer takes 20-60s per iteration vs <1s.
# We try installing; if it fails we warn loudly but continue (fallback exists).
print("\nInstalling gsplat CUDA rasterizer...")
t1 = time.time()

gsplat_ok = pip_install("gsplat")
if gsplat_ok:
    print(f"  gsplat installed in {time.time()-t1:.0f}s")
else:
    print("⚠️  gsplat install failed — will use slower Python fallback.")
    print("   Training will work but will be significantly slower.")

# ── 2c. Validate all imports ─────────────────────────────────────────────────
print("\nValidating imports...")

validation_failures = []

required_imports = [
    ("torch",   "PyTorch"),
    ("PIL",     "Pillow"),
    ("yaml",    "PyYAML"),
    ("plyfile", "plyfile"),
    ("tqdm",    "tqdm"),
    ("numpy",   "NumPy"),
    ("scipy",   "SciPy"),
    ("imageio", "imageio"),
]

for mod_name, display_name in required_imports:
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✅  {display_name:<12} {ver}")
    except ImportError as e:
        print(f"  ❌  {display_name:<12} MISSING — {e}")
        validation_failures.append(display_name)

# Optional but important: gsplat
GSPLAT_AVAILABLE = False
try:
    import gsplat
    GSPLAT_VERSION = getattr(gsplat, "__version__", "unknown")
    GSPLAT_AVAILABLE = True
    print(f"  ✅  gsplat        {GSPLAT_VERSION} (fast CUDA rasterizer)")
except ImportError:
    print(f"  ⚠️  gsplat        NOT AVAILABLE — using Python fallback (slow)")

if validation_failures:
    raise RuntimeError(
        f"Required imports missing: {validation_failures}\n"
        "Try: Runtime → Factory reset runtime → run from Cell 1."
    )


# ── 2e. gsplat version compatibility check ───────────────────────────────────
if GSPLAT_AVAILABLE:
    import re
    _ver_parts = GSPLAT_VERSION.split(".")
    try:
        _major, _minor = int(_ver_parts[0]), int(_ver_parts[1])
        if _major < 1 or (_major == 1 and _minor < 4):
            import warnings
            warnings.warn(f"gsplat {GSPLAT_VERSION} is older than tested minimum 1.4.x — instability possible.")
        elif _major > 1:
            import warnings
            warnings.warn(f"gsplat {GSPLAT_VERSION} is newer than tested maximum 1.x — API may have changed.")
        else:
            print(f"  ✅  gsplat version {GSPLAT_VERSION} is within the tested range (1.4.x – 1.x.x)")
    except (ValueError, IndexError):
        print(f"  ⚠️  Cannot parse gsplat version '{GSPLAT_VERSION}'")

# ── 2d. Final environment summary ────────────────────────────────────────────
import torch
print("\n" + "═" * 50)
print("  ENVIRONMENT SUMMARY")
print("═" * 50)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.version.cuda}")
print(f"  cuDNN    : {torch.backends.cudnn.version()}")
print(f"  GPU      : {torch.cuda.get_device_name(0)}")
print(f"  VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  gsplat   : {'✅ available' if GSPLAT_AVAILABLE else '⚠️ unavailable (slow fallback)'}")
print("═" * 50)
print("\n→  Run Cell 3 to mount Google Drive.")


---
## Cell 3 — Google Drive Integration

Mounts Google Drive and creates the persistent folder structure.  
All checkpoints, previews, logs, and exports are synced here automatically.  
If Colab crashes, your checkpoint survives in Drive.

**Folder structure created:**
```
/drive/MyDrive/MonoSplat/
├── datasets/      ← optional: pre-stage your zip here
├── checkpoints/   ← auto-saved every N iterations
├── previews/      ← rendered preview images
├── exports/       ← final .splat, .ply, logs
└── logs/          ← metrics.json, training_log.txt
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 3 — GOOGLE DRIVE INTEGRATION
#  Purpose : Mount Drive for crash-safe persistence of all training artifacts.
#  Why    : Colab sessions can disconnect at any time. Anything saved only
#           in /content is lost on reconnect. Drive persists across sessions.
# ══════════════════════════════════════════════════════════════════════════════

import os
from pathlib import Path

DRIVE_ROOT = Path("/drive/MyDrive/MonoSplat")
DRIVE_MOUNTED = False

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    print("(A Google sign-in popup will appear — authorize to continue)")
    drive.mount("/drive", force_remount=False)
    DRIVE_MOUNTED = True
    print("✅  Drive mounted at /drive")
except Exception as e:
    print(f"⚠️  Drive mount failed: {e}")
    print("   Training will continue, but checkpoints will NOT be saved to Drive.")
    print("   If Colab disconnects you will lose progress.")

# ── 3a. Create MonoSplat folder structure ────────────────────────────────────
DRIVE_DIRS = {
    "datasets":    DRIVE_ROOT / "datasets",
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "previews":    DRIVE_ROOT / "previews",
    "exports":     DRIVE_ROOT / "exports",
    "logs":        DRIVE_ROOT / "logs",
}

if DRIVE_MOUNTED:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for name, path in DRIVE_DIRS.items():
        path.mkdir(parents=True, exist_ok=True)
        print(f"  ✅  /MonoSplat/{name}/")
    print(f"\n✅  Drive workspace ready at: {DRIVE_ROOT}")
else:
    print("\n⚠️  Using local /content only — no Drive backup.")

# ── 3b. Helper function for Drive sync ──────────────────────────────────────
import shutil

def sync_to_drive(src_path: str, dest_subdir: str = "checkpoints") -> bool:
    """
    Copy a file to the corresponding Drive subdirectory.
    Returns True on success, False on failure (non-fatal).
    """
    if not DRIVE_MOUNTED:
        return False
    try:
        dest = DRIVE_DIRS[dest_subdir] / Path(src_path).name
        shutil.copy2(src_path, dest)
        return True
    except Exception as e:
        print(f"  ⚠️  Drive sync failed: {e}")
        return False

print("\n→  Run Cell 4 to select your hardware preset.")


---
## Cell 4 — Hardware Preset Selection

Auto-detects your GPU and applies the appropriate preset.  
You can override any value below, but start with the auto-detected preset —  
it's tuned to keep VRAM usage below 80% under worst-case densification.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 4 — HARDWARE PRESET SELECTION
#  Purpose : Auto-select training parameters based on detected GPU.
#  Why     : Aggressive settings on T4 cause OOM, NaN cascades, and kernel
#            crashes. Conservative presets trade some quality for stability.
#  Override: Change values in USER OVERRIDES section below.
# ══════════════════════════════════════════════════════════════════════════════

import torch

props    = torch.cuda.get_device_properties(0)
VRAM_GB  = props.total_memory / 1e9
GPU_NAME = props.name

# ── 4a. Hardware presets ─────────────────────────────────────────────────────
# Each preset is tuned to keep peak VRAM < 80% of total.
# T4 is the hardest: 15.6 GB sounds like a lot but densification
# temporarily doubles Gaussian count, so headroom matters.

PRESETS = {
    "T4_conservative": {
        "description":       "T4 — stable, conservative (free Colab)",
        "resolution":        (640, 360),
        "sh_degree":         3,           # SH0=flat color, SH1=view-dep, SH3=full
        "iterations":        10000,        # conservative; push to 5000 if stable
        "max_gaussians":     250_000,     # hard cap; densification stops here
        "max_init_points":   125_000,      # initial cloud downsampled to this
        "densify_from":      500,
        "densify_until":     10000,
        "densify_interval":  50,          # less frequent = more stable
        "grad_threshold":    0.0001,      # higher = fewer, safer clones
        "save_interval":     250,
        "preview_interval":  500,
        "vram_warn_gb":      12.0,        # warn when allocated > this
        "vram_abort_gb":     14.0,        # emergency prune when > this
        "batch_size":        1,           # images rendered per step
        "percent_dense":     0.01,
    },
    "L4_balanced": {
        "description":       "L4 — balanced quality/stability (Colab Pro)",
        "resolution":        (960, 540),
        "sh_degree":         3,
        "iterations":        12000,
        "max_gaussians":     200_000,
        "max_init_points":   125_000,
        "densify_from":      500,
        "densify_until":     10000,
        "densify_interval":  50, 
        "grad_threshold":    0.0001,
        "save_interval":     500,
        "preview_interval":  1000,
        "vram_warn_gb":      18.0,
        "vram_abort_gb":     20.0,
        "batch_size":        1,
        "percent_dense":     0.01,
    },
    "A100_high_quality": {
        "description":       "A100 — high quality (Colab Pro+)",
        "resolution":        (1280, 720),
        "sh_degree":         3,
        "iterations":        15000,
        "max_gaussians":     500_000,
        "max_init_points":   250_000,
        "densify_from":      500,
        "densify_until":     10000,
        "densify_interval":  50, 
        "grad_threshold":    0.0001,
        "save_interval":     1000,
        "preview_interval":  2000,
        "vram_warn_gb":      32.0,
        "vram_abort_gb":     38.0,
        "batch_size":        1,
        "percent_dense":     0.01,
    },
}

# ── 4b. Auto-select preset ───────────────────────────────────────────────────
if VRAM_GB >= 35:
    PRESET_NAME = "A100_high_quality"
elif VRAM_GB >= 20:
    PRESET_NAME = "L4_balanced"
else:
    PRESET_NAME = "T4_conservative"

CFG = PRESETS[PRESET_NAME].copy()

# ── 4c. USER OVERRIDES — edit here to customize ─────────────────────────────
# Uncomment and change any value. Start conservative and increase only
# if previous run was stable (no OOM, no NaN, Gaussian count stayed below cap).

# CFG["iterations"]    = 5000       # more = better quality, more time
# CFG["max_gaussians"] = 75_000     # lower if you get OOM
# CFG["resolution"]    = (480, 270) # lower if very large scene
# CFG["sh_degree"]     = 0          # SH0 = flat color, most stable

# ── 4d. Extract config into module-level variables ───────────────────────────
W, H               = CFG["resolution"]
SH_DEGREE          = CFG["sh_degree"]
ITERATIONS         = CFG["iterations"]
MAX_GAUSSIANS      = CFG["max_gaussians"]
MAX_INIT_POINTS    = CFG["max_init_points"]
DENSIFY_FROM       = CFG["densify_from"]
DENSIFY_UNTIL      = CFG["densify_until"]
DENSIFY_INTERVAL   = CFG["densify_interval"]
GRAD_THRESHOLD     = CFG["grad_threshold"]
SAVE_INTERVAL      = CFG["save_interval"]
PREVIEW_INTERVAL   = CFG["preview_interval"]
VRAM_WARN_GB       = CFG["vram_warn_gb"]
VRAM_ABORT_GB      = CFG["vram_abort_gb"]
BATCH_SIZE         = CFG["batch_size"]
PERCENT_DENSE      = CFG["percent_dense"]

# ── 4e. Print final configuration ────────────────────────────────────────────
print("═" * 55)
print(f"  HARDWARE PRESET: {PRESET_NAME}")
print("═" * 55)
print(f"  GPU             : {GPU_NAME}  ({VRAM_GB:.1f} GB)")
print(f"  Resolution      : {W}×{H}")
print(f"  SH degree       : {SH_DEGREE}")
print(f"  Iterations      : {ITERATIONS:,}")
print(f"  Max Gaussians   : {MAX_GAUSSIANS:,}  (hard cap — densification stops here)")
print(f"  Init points cap : {MAX_INIT_POINTS:,}  (downsampled if larger)")
print(f"  Densify range   : iter {DENSIFY_FROM}–{DENSIFY_UNTIL}, every {DENSIFY_INTERVAL}")
print(f"  Grad threshold  : {GRAD_THRESHOLD}")
print(f"  Save interval   : every {SAVE_INTERVAL} iterations")
print(f"  Preview interval: every {PREVIEW_INTERVAL} iterations")
print(f"  VRAM warn at    : {VRAM_WARN_GB} GB allocated")
print(f"  VRAM abort at   : {VRAM_ABORT_GB} GB → emergency prune")
print("═" * 55)
print("\n→  Run Cell 5 to upload your dataset.")


---
## Cell 5 — Dataset Upload

Upload your `monosplat_job.zip` containing:
```
work/<job_id>/frames/          ← PNG frames from FFmpeg
work/<job_id>/colmap/sparse_text/  ← cameras.txt, images.txt, points3D.txt
config/config.yaml
```

**Alternative**: Place the zip in `/drive/MyDrive/MonoSplat/datasets/` and set `DRIVE_DATASET = True` below.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 5 — DATASET UPLOAD
#  Purpose : Load dataset from upload or Google Drive.
#  Options :
#    DRIVE_DATASET = False → Colab file upload dialog (default)
#    DRIVE_DATASET = True  → Read zip directly from Drive
#  Why separate from validation? Separation makes it easy to re-run
#  validation without re-uploading 200 MB.
# ══════════════════════════════════════════════════════════════════════════════

import os, zipfile, glob, shutil
from pathlib import Path

# ── USER OPTION ──────────────────────────────────────────────────────────────
# Set to True if your zip is already in /drive/MyDrive/MonoSplat/datasets/
DRIVE_DATASET = False
# If DRIVE_DATASET = True, set the filename here:
DRIVE_DATASET_FILENAME = "monosplat_job.zip"  # filename in Drive/datasets/
# ─────────────────────────────────────────────────────────────────────────────

os.chdir("/content")
EXTRACT_DIR = Path("/content/monosplat")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = None

if DRIVE_DATASET and DRIVE_MOUNTED:
    # ── Load from Drive ──────────────────────────────────────────────────────
    drive_zip = DRIVE_DIRS["datasets"] / DRIVE_DATASET_FILENAME
    if not drive_zip.exists():
        raise FileNotFoundError(
            f"❌  Dataset not found in Drive at: {drive_zip}\n"
            f"   Place your zip at: /drive/MyDrive/MonoSplat/datasets/{DRIVE_DATASET_FILENAME}\n"
            "   OR set DRIVE_DATASET = False and upload manually."
        )
    zip_path = str(drive_zip)
    print(f"📦  Using Drive dataset: {zip_path}")
    print(f"    Size: {os.path.getsize(zip_path)/1e6:.1f} MB")

else:
    # ── Upload dialog ────────────────────────────────────────────────────────
    from google.colab import files
    print("📂  File picker opening...")
    print("    Select your monosplat_job.zip")
    print("    (Upload speed: ~50–100 MB/min — please wait)")
    print()

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Re-run this cell and select your zip.")

    fname = list(uploaded.keys())[0]
    src   = Path("/content") / fname
    dst   = EXTRACT_DIR / fname
    shutil.move(str(src), str(dst))
    zip_path = str(dst)
    print(f"\n📦  Uploaded: {fname}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

# ── 5a. Extract zip ──────────────────────────────────────────────────────────
print(f"\n📂  Extracting to {EXTRACT_DIR} ...")
try:
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("✅  Extraction complete")
except zipfile.BadZipFile:
    raise RuntimeError(
        f"❌  {zip_path} is not a valid zip file.\n"
        "   Re-create with: python scripts/zip_for_colab.py <job_id>"
    )

os.chdir(EXTRACT_DIR)

# ── 5b. Auto-detect job_id ───────────────────────────────────────────────────
work_dirs = sorted([
    d for d in glob.glob(str(EXTRACT_DIR / "work" / "*"))
    if os.path.isdir(d)
])

if not work_dirs:
    raise RuntimeError(
        "❌  No work/<job_id>/ directory found in zip.\n"
        "   Expected: work/<job_id>/frames/  and  work/<job_id>/colmap/sparse_text/\n"
        "   Re-create with: python scripts/zip_for_colab.py <job_id>"
    )

JOB_ID      = os.path.basename(work_dirs[0])
FRAMES_DIR  = str(EXTRACT_DIR / f"work/{JOB_ID}/frames")
COLMAP_DIR  = str(EXTRACT_DIR / f"work/{JOB_ID}/colmap/sparse_text")
OUTPUT_DIR  = str(EXTRACT_DIR / f"work/{JOB_ID}/models/gaussian")
CKPT_DIR    = str(EXTRACT_DIR / f"work/{JOB_ID}/checkpoints")
CONFIG_PATH = str(EXTRACT_DIR / "config/config.yaml")
PREVIEW_DIR = str(EXTRACT_DIR / f"work/{JOB_ID}/previews")
LOG_DIR     = str(EXTRACT_DIR / f"work/{JOB_ID}/logs")

for d in [OUTPUT_DIR, CKPT_DIR, PREVIEW_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

frame_count = len(glob.glob(os.path.join(FRAMES_DIR, "*.png"))) + \
              len(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))

print(f"\n✅  Job ID     : {JOB_ID}")
print(f"   Frames dir : {FRAMES_DIR}  ({frame_count} files)")
print(f"   COLMAP dir : {COLMAP_DIR}")
print(f"   Output dir : {OUTPUT_DIR}")
print(f"   Config     : {CONFIG_PATH}")
print("\n→  Run Cell 6 to validate dataset.")


---
## Cell 6 — Dataset Validation

A dedicated validation stage that **hard-fails** on bad data rather than letting
training proceed with corrupted or incomplete input.  

Bad data doesn't produce a bad result — it produces a kernel crash 20 minutes
into training. Catching it here saves time and frustration.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 6 — DATASET VALIDATION
#  Purpose : Validate all required files before touching GPU.
#  Philosophy: Fail fast with clear diagnostics rather than cryptic crashes.
# ══════════════════════════════════════════════════════════════════════════════

import os, glob
import numpy as np
from PIL import Image

validation_errors   = []
validation_warnings = []

print("═" * 55)
print("  DATASET VALIDATION")
print("═" * 55)

# ── 6a. COLMAP files ─────────────────────────────────────────────────────────
print("\n[1/4] COLMAP files:")
colmap_files = {
    "cameras.txt":  {"min_lines": 1,     "desc": "camera intrinsics"},
    "images.txt":   {"min_lines": 2,     "desc": "image poses"},
    "points3D.txt": {"min_lines": 100,   "desc": "3D point cloud"},
}

points3d_count = 0
image_pose_count = 0

for fname, spec in colmap_files.items():
    path = os.path.join(COLMAP_DIR, fname)
    if not os.path.exists(path):
        msg = f"MISSING: {fname} — {spec['desc']}"
        print(f"  ❌  {msg}")
        validation_errors.append(msg)
        continue

    with open(path) as f:
        data_lines = [l for l in f if not l.startswith("#") and l.strip()]

    count = len(data_lines)
    if count < spec["min_lines"]:
        msg = f"{fname} has only {count} data lines (min {spec['min_lines']})"
        print(f"  ❌  {msg}")
        validation_errors.append(msg)
    else:
        print(f"  ✅  {fname:<20}  {count:>8,} lines")

    if fname == "points3D.txt":
        points3d_count = count
    if fname == "images.txt":
        # images.txt has 2 lines per image (header + keypoints)
        image_pose_count = count // 2

# ── 6b. Frame images ─────────────────────────────────────────────────────────
print("\n[2/4] Frame images:")
frame_files = sorted(
    glob.glob(os.path.join(FRAMES_DIR, "*.png")) +
    glob.glob(os.path.join(FRAMES_DIR, "*.jpg")) +
    glob.glob(os.path.join(FRAMES_DIR, "*.jpeg"))
)

if not frame_files:
    validation_errors.append(f"No images found in {FRAMES_DIR}")
    print(f"  ❌  No images in {FRAMES_DIR}")
elif len(frame_files) < 10:
    validation_errors.append(f"Only {len(frame_files)} frames — minimum 10 required")
    print(f"  ❌  Only {len(frame_files)} frames (minimum 10 required for reconstruction)")
elif len(frame_files) < 30:
    msg = f"Only {len(frame_files)} frames — reconstruction may be sparse"
    validation_warnings.append(msg)
    print(f"  ⚠️  {msg}")
else:
    print(f"  ✅  {len(frame_files)} frames found")

# Validate a sample of images are readable and consistent
if frame_files:
    sample = frame_files[:min(5, len(frame_files))]
    resolutions = set()
    corrupt = []
    for fp in sample:
        try:
            img = Image.open(fp)
            resolutions.add(img.size)
        except Exception:
            corrupt.append(fp)

    if corrupt:
        validation_errors.append(f"Corrupted images: {corrupt}")
        print(f"  ❌  Corrupted images: {corrupt}")
    else:
        res_str = ", ".join(f"{w}×{h}" for w, h in resolutions)
        print(f"  ✅  Image resolution(s): {res_str}")
        if len(resolutions) > 1:
            validation_warnings.append("Mixed resolutions detected — COLMAP may be unreliable")
            print(f"  ⚠️  Mixed resolutions — training may have artifacts")

# ── 6c. Config file ───────────────────────────────────────────────────────────
print("\n[3/4] Config:")
if not os.path.exists(CONFIG_PATH):
    validation_warnings.append("config.yaml missing — will use defaults")
    print(f"  ⚠️  {CONFIG_PATH} not found — using built-in defaults")
else:
    print(f"  ✅  {CONFIG_PATH}")

# ── 6d. Registration ratio ────────────────────────────────────────────────────
print("\n[4/4] Reconstruction quality:")
if len(frame_files) > 0 and image_pose_count > 0:
    reg_ratio = image_pose_count / len(frame_files)
    print(f"  📊  Registration ratio: {image_pose_count}/{len(frame_files)} = {reg_ratio:.1%}")
    if reg_ratio < 0.5:
        msg = f"Registration ratio {reg_ratio:.1%} is very low — COLMAP found poses for < half the frames"
        validation_warnings.append(msg)
        print(f"  ⚠️  Low registration — training may converge poorly")
    elif reg_ratio < 0.8:
        print(f"  ⚠️  Moderate registration — acceptable but not ideal")
    else:
        print(f"  ✅  Good registration ratio")

print(f"  📊  3D points: {points3d_count:,}")
if points3d_count < 1000:
    msg = f"Only {points3d_count:,} 3D points — scene may be too sparse for good reconstruction"
    validation_warnings.append(msg)
    print(f"  ⚠️  {msg}")
elif points3d_count < 10_000:
    print(f"  ⚠️  Sparse point cloud — quality may be limited")
else:
    print(f"  ✅  Dense enough point cloud")

# ── 6e. Summary ───────────────────────────────────────────────────────────────
print("\n" + "═" * 55)
if validation_errors:
    print("  ❌  VALIDATION FAILED — fix these errors before training:")
    for err in validation_errors:
        print(f"     • {err}")
    print("═" * 55)
    raise SystemExit("Dataset validation failed. Fix errors and re-run.")
elif validation_warnings:
    print(f"  ⚠️  VALIDATION PASSED WITH WARNINGS ({len(validation_warnings)}):")
    for w in validation_warnings:
        print(f"     • {w}")
    print("\n  Training can proceed. Warnings are informational.")
else:
    print("  ✅  VALIDATION PASSED — dataset looks clean")
print("═" * 55)

# ── 6e. Image quality analysis (blur + exposure) ─────────────────────────────
# Runs before training to catch dataset problems early.
# This uses src.diagnostics.dataset_quality if available.
print("\n[5/5] Image quality analysis:")
try:
    import sys, os
    _repo_path = os.path.join(COLAB_BASE, "monosplat")
    if _repo_path not in sys.path:
        sys.path.insert(0, _repo_path)
    from src.diagnostics.dataset_quality import check_blur, check_exposure_consistency
    _blur_result = check_blur(frame_files, threshold=50.0, sample_n=20)
    _expo_result = check_exposure_consistency(frame_files, max_std_pct=30.0, sample_n=20)
    for _res in [_blur_result, _expo_result]:
        if _res.get("skipped"):
            print(f"  ⏭️  {_res['check']}: skipped ({_res['message']})")
        elif _res["ok"]:
            print(f"  ✅  {_res['check']}: {_res['message']}")
        else:
            print(f"  ⚠️  {_res['check']}: {_res['message']}")
            validation_warnings.append(_res["message"])
except Exception as _qe:
    print(f"  ⏭️  Image quality analysis skipped: {_qe}")

print("\n→  Run Cell 7 to fetch source code and initialize Gaussians.")


---
## Cell 7 — Source Code + Safe Gaussian Initialization

Fetches the MonoSplat source from GitHub and initializes the Gaussian model.  
Initialization is conservative: the point cloud is downsampled to `MAX_INIT_POINTS`,  
scales are clamped, and every tensor is validated for NaN/Inf before training begins.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 7 — SOURCE CODE + SAFE GAUSSIAN INITIALIZATION
#  Purpose : Fetch src/, initialize Gaussian model, validate tensors.
#  Why safe init matters: starting with NaN scales or extreme positions
#  causes loss explosions in the FIRST iteration — before any checkpoint
#  is saved, making the failure impossible to recover from.
# ══════════════════════════════════════════════════════════════════════════════

import os, sys, subprocess, shutil, gc, random
import numpy as np
import torch

# ── 7a. Deterministic seed ────────────────────────────────────────────────────
# Set seeds BEFORE any model creation or data shuffling.
# Same seed = same initialization = reproducible results.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Note: full determinism requires CUBLAS_WORKSPACE_CONFIG but has perf cost.
# We trade perfect determinism for practical reproducibility here.
torch.backends.cudnn.deterministic = False  # True kills performance on T4
torch.backends.cudnn.benchmark     = True   # auto-tune convolution kernels
print(f"✅  Global seed: {SEED}")

# ── 7b. Fetch source code ─────────────────────────────────────────────────────
REPO_URL = "https://github.com/Somaskandan931/monosplat.git"
TMP_DIR  = "/tmp/monosplat_src"
CWD      = os.getcwd()
src_dir  = os.path.join(CWD, "src")
scr_dir  = os.path.join(CWD, "scripts")

for d in [TMP_DIR, src_dir, scr_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)

print(f"\n📦  Cloning {REPO_URL} (sparse: src/ + scripts/)...")
r = subprocess.run(
    ["git", "clone", "--filter=blob:none", "--sparse", REPO_URL, TMP_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    print(f"STDERR: {r.stderr[-2000:]}")
    raise RuntimeError(
        "❌  Git clone failed.\n"
        "   Check that https://github.com/Somaskandan931/monosplat is public.\n"
        "   Check your internet connection."
    )

subprocess.run(["git", "-C", TMP_DIR, "sparse-checkout", "set", "src", "scripts"],
               capture_output=True)

if not os.path.isdir(os.path.join(TMP_DIR, "src")):
    raise RuntimeError("❌  src/ not found in cloned repo. Check repo structure.")

shutil.copytree(os.path.join(TMP_DIR, "src"),     src_dir)
shutil.copytree(os.path.join(TMP_DIR, "scripts"), scr_dir, dirs_exist_ok=True)
print(f"✅  src/ copied to: {src_dir}")

if CWD not in sys.path:
    sys.path.insert(0, CWD)

# ── 7c. Import validation ─────────────────────────────────────────────────────
print("\nValidating src/ imports...")
try:
    from src.utils.config_loader            import load_config
    from src.preprocessing.utils           import load_colmap_model
    from src.reconstruction.gaussian_model import GaussianModel
    from src.reconstruction.trainer        import GaussianTrainer
    from src.reconstruction.loss           import combined_loss, psnr_metric
    from src.renderer.camera               import Camera as ViewerCamera
    from src.renderer.renderer             import GaussianRenderer
    print("✅  All src/ imports OK")
except ImportError as e:
    raise ImportError(
        f"❌  Import failed: {e}\n"
        "   Re-run this cell to re-clone src/."
    )

# ── 7d. Load config ───────────────────────────────────────────────────────────
if os.path.exists(CONFIG_PATH):
    cfg = load_config(CONFIG_PATH)
    print(f"✅  Config loaded: {CONFIG_PATH}")
else:
    cfg = load_config(None)  # use defaults
    print("⚠️  Using default config (config.yaml not found)")

# ── 7e. Load COLMAP ───────────────────────────────────────────────────────────
print(f"\n📂  Loading COLMAP from: {COLMAP_DIR}")
cameras_colmap, images_colmap, points3d = load_colmap_model(COLMAP_DIR)
print(f"✅  {len(cameras_colmap)} cameras, {len(images_colmap)} images, {len(points3d):,} points")

# ── 7f. Build training cameras + images ──────────────────────────────────────
from pathlib import Path
from PIL import Image

print(f"\n🖼️   Loading {len(images_colmap)} training images at {W}×{H}...")
train_cameras, train_images, missing = [], [], []

for img_data in sorted(images_colmap.values(), key=lambda i: i.name):
    cam_data = cameras_colmap[img_data.camera_id]
    cam      = ViewerCamera.from_colmap(img_data, cam_data, W, H)

    img_path = Path(FRAMES_DIR) / img_data.name
    if not img_path.exists():
        img_path = Path(FRAMES_DIR) / Path(img_data.name).name
    if not img_path.exists():
        missing.append(img_data.name)
        continue

    img_np = np.array(
        (lambda _img: _img.resize((W, H), Image.LANCZOS) if (W, H) != _img.size else _img)(Image.open(img_path).convert("RGB")),
        dtype=np.float32
    ) / 255.0
    train_cameras.append(cam)
    # Store on CPU — move to GPU per-iteration to avoid filling VRAM
    train_images.append(torch.from_numpy(img_np).permute(2, 0, 1).cpu())

if missing:
    print(f"  ⚠️  {len(missing)} frames missing (first 3: {missing[:3]})")
if not train_cameras:
    raise RuntimeError(f"❌  No cameras loaded. Check FRAMES_DIR={FRAMES_DIR}")
print(f"✅  {len(train_cameras)} cameras loaded")

# ── 7g. Safe point cloud initialization ──────────────────────────────────────
# This is the most important initialization safety step.
# 1. Downsample to MAX_INIT_POINTS to prevent OOM at init
# 2. Clamp positions to finite range
# 3. Validate all tensors

print(f"\n🔷  Initializing Gaussian cloud...")
xyzs = np.array([p.xyz for p in points3d.values()], dtype=np.float32)
rgbs = np.array([p.rgb for p in points3d.values()], dtype=np.float32) / 255.0

print(f"  Raw cloud: {len(xyzs):,} points")

# Downsample if exceeds cap
if len(xyzs) > MAX_INIT_POINTS:
    idx  = np.random.choice(len(xyzs), MAX_INIT_POINTS, replace=False)
    xyzs = xyzs[idx]
    rgbs = rgbs[idx]
    print(f"  ⚠️  Downsampled to {MAX_INIT_POINTS:,} points (cap)")

# Clamp XYZ to remove outliers (COLMAP sometimes has extreme outliers)
# Outlier Gaussians at ±1000 units cause scale explosions immediately.
P99 = np.percentile(np.abs(xyzs), 99, axis=0)
clamp_val = float(np.max(P99) * 2.0)   # 2× the 99th percentile
before = len(xyzs)
valid_mask = np.all(np.abs(xyzs) < clamp_val, axis=1)
xyzs = xyzs[valid_mask]
rgbs = rgbs[valid_mask]
removed = before - len(xyzs)
if removed > 0:
    print(f"  ⚠️  Removed {removed} outlier points (>{clamp_val:.1f} units from origin)")

# Validate no NaN/Inf in input data
assert not np.any(np.isnan(xyzs)), "❌  NaN in point positions after clamping"
assert not np.any(np.isinf(xyzs)), "❌  Inf in point positions after clamping"
assert not np.any(np.isnan(rgbs)), "❌  NaN in point colors"
print(f"  ✅  {len(xyzs):,} clean points validated")

# ── 7h. Create Gaussian model ─────────────────────────────────────────────────
device = "cuda"
model  = GaussianModel(sh_degree=SH_DEGREE)
model  = model.to(device)
model.create_from_points(xyzs, rgbs)

# ── 7i. Validate model tensors ────────────────────────────────────────────────
print("\n  Validating model tensors...")
tensor_checks = {
    "positions":  model._positions,
    "scales":     model._scales,
    "rotations":  model._rotations,
    "opacities":  model._opacities,
}

for name, tensor in tensor_checks.items():
    has_nan = torch.isnan(tensor).any().item()
    has_inf = torch.isinf(tensor).any().item()
    status  = "❌ NaN" if has_nan else ("❌ Inf" if has_inf else "✅")
    print(f"    {name:<12}  shape={tuple(tensor.shape)}  {status}")
    if has_nan or has_inf:
        raise RuntimeError(f"Invalid tensor '{name}' after initialization. "
                           "Check input data quality.")

# ── 7j. VRAM snapshot ─────────────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()
alloc_gb = torch.cuda.memory_allocated() / 1e9
resrv_gb = torch.cuda.memory_reserved()  / 1e9
free_gb  = torch.cuda.mem_get_info()[0]  / 1e9

print(f"\n📊  Post-init VRAM:")
print(f"    Allocated : {alloc_gb:.2f} GB")
print(f"    Reserved  : {resrv_gb:.2f} GB")
print(f"    Free      : {free_gb:.2f} GB")
print(f"    Gaussians : {len(model):,}")

print("\n✅  Gaussian model initialized safely.")
print("\n→  Run Cell 8 to start training.")


---
## Cell 8 — VRAM-Aware Training Loop

The training loop is the most critical cell. Key safety features:

- **Hard Gaussian cap**: densification halts when `MAX_GAUSSIANS` is reached
- **VRAM monitoring**: every iteration logs allocated/reserved memory
- **Emergency pruning**: if VRAM exceeds `VRAM_ABORT_GB`, prunes lowest-opacity Gaussians
- **Pre-densification checkpoints**: saved before each densification step
- **NaN detection**: NaN losses are skipped with counter; too many = abort
- **Auto-resume**: detects existing checkpoints and resumes from latest
- **Drive sync**: every checkpoint is copied to Drive immediately

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 8 — VRAM-AWARE TRAINING LOOP
#  Purpose : Full training with VRAM monitoring, checkpointing, crash protection.
#  Resume  : Automatically resumes from the latest checkpoint if one exists.
# ══════════════════════════════════════════════════════════════════════════════

import os, gc, json, time, glob, shutil
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

# ── 8a. Renderer setup ────────────────────────────────────────────────────────
torch.cuda.empty_cache(); gc.collect()

renderer = GaussianRenderer(
    width=W, height=H,
    bg_color=getattr(cfg.renderer, "background_color", [0, 0, 0]),
    device=device,
    batch_size=BATCH_SIZE,
    use_gsplat=GSPLAT_AVAILABLE,
)

trainer = GaussianTrainer(
    model=model,
    renderer=renderer.render_torch,
    train_cameras=train_cameras,
    train_images=train_images,
    cfg=cfg,
)

optimizer    = trainer.optimizer
lambda_dssim = getattr(cfg.training, "lambda_dssim", 0.2)

# ── 8b. Auto-resume: detect existing checkpoint ───────────────────────────────
# Checkpoints are saved as iter_{N:06d}.ckpt
# On resume, we pick the latest and continue from that iteration.
START_ITER = 0
existing_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "iter_*.ckpt")))

if existing_ckpts:
    latest_ckpt = existing_ckpts[-1]
    print(f"📂  Found checkpoint: {latest_ckpt}")
    try:
        ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        START_ITER = ckpt["iteration"] + 1
        print(f"✅  Resuming from iteration {START_ITER:,}")
        print(f"    Checkpoint loss: {ckpt.get('loss', 'N/A')}")
        print(f"    Checkpoint Gaussians: {ckpt.get('n_gaussians', 'N/A'):,}")
    except Exception as e:
        print(f"⚠️  Checkpoint load failed: {e}")
        print("   Starting from scratch.")
        START_ITER = 0
else:
    print("ℹ️  No checkpoint found — starting from scratch.")

# ── 8c. Checkpoint save function ─────────────────────────────────────────────
def save_checkpoint(iteration: int, loss: float, label: str = "") -> str:
    """
    Save a full training checkpoint and sync to Drive.
    Returns the checkpoint path.
    """
    tag = f"_{label}" if label else ""
    ckpt_file = os.path.join(CKPT_DIR, f"iter_{iteration:06d}{tag}.ckpt")
    ply_file  = os.path.join(OUTPUT_DIR, f"iter_{iteration:06d}{tag}.ply")

    payload = {
        "iteration":       iteration,
        "loss":            loss,
        "n_gaussians":     len(model),
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "config": {
            "iterations":       ITERATIONS,
            "max_gaussians":    MAX_GAUSSIANS,
            "resolution":       (W, H),
            "sh_degree":        SH_DEGREE,
            "densify_from":     DENSIFY_FROM,
            "densify_until":    DENSIFY_UNTIL,
            "densify_interval": DENSIFY_INTERVAL,
            "grad_threshold":   GRAD_THRESHOLD,
        }
    }

    try:
        torch.save(payload, ckpt_file)
        trainer._save(iteration)   # also save .ply via trainer's built-in method
        sync_to_drive(ckpt_file, "checkpoints")
        if os.path.exists(ply_file):
            sync_to_drive(ply_file, "checkpoints")
    except Exception as e:
        print(f"  ⚠️  Checkpoint save error: {e}")

    return ckpt_file

# ── 8d. VRAM monitor function ────────────────────────────────────────────────
def vram_stats() -> dict:
    """Return current VRAM stats as a dict (all in GB).

    IMPORTANT: torch.cuda.synchronize() is called first to ensure all
    pending CUDA operations have completed before the counters are read.
    Without this, memory_allocated() often returns 0.0 during training.
    """
    if not torch.cuda.is_available():
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "peak_gb": 0.0, "free_gb": 0.0}
    try:
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated()
        reserved  = torch.cuda.memory_reserved()
        peak      = torch.cuda.max_memory_allocated()
        free_b, _ = torch.cuda.mem_get_info()
        return {
            "allocated_gb": round(allocated / 1e9, 2),
            "reserved_gb":  round(reserved  / 1e9, 2),
            "peak_gb":      round(peak      / 1e9, 2),
            "free_gb":      round(free_b    / 1e9, 2),
        }
    except Exception:
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "peak_gb": 0.0, "free_gb": 0.0}

# ── 8e. Densification safety function ────────────────────────────────────────
def safe_densify(grad_accum, grad_denom, iteration):
    """
    Perform densification with full safety guards:
    1. Pre-densification checkpoint
    2. Gaussian count cap enforcement
    3. Post-densification NaN/Inf validation
    4. Rollback on failure
    Returns (new_n, success).
    """
    n_before = len(model)

    # Skip if already at cap
    if n_before >= MAX_GAUSSIANS:
        return n_before, False

    # Save a pre-densification checkpoint for rollback
    pre_ckpt = os.path.join(CKPT_DIR, f"pre_densify_{iteration:06d}.ckpt")
    try:
        torch.save({"model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict()}, pre_ckpt)
    except Exception:
        pass  # pre-checkpoint failure is non-fatal

    try:
        trainer._grad_accum = grad_accum.clone()
        trainer._grad_denom = grad_denom.clone()
        trainer._densify_and_prune(
            grad_threshold=GRAD_THRESHOLD,
            percent_dense=PERCENT_DENSE
        )
    except Exception as e:
        tqdm.write(f"  ⚠️  Densification failed: {e} — rolling back")
        # Attempt rollback
        try:
            ckpt = torch.load(pre_ckpt, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state"])
            trainer._setup_optimizer()
        except Exception:
            pass
        return n_before, False

    n_after = len(model)

    # Enforce MAX_GAUSSIANS cap by pruning lowest-opacity Gaussians
    if n_after > MAX_GAUSSIANS:
        excess = n_after - MAX_GAUSSIANS
        with torch.no_grad():
            _, low_idx = model.opacities.squeeze().topk(excess, largest=False)
            mask = torch.zeros(n_after, dtype=torch.bool, device=device)
            mask[low_idx] = True
        model.prune_points(mask)
        tqdm.write(f"    Cap enforced: pruned {excess:,} low-opacity Gaussians")
        n_after = len(model)

    # Post-densification tensor validation
    for attr_name in ["_positions", "_scales", "_rotations", "_opacities"]:
        tensor = getattr(model, attr_name, None)
        if tensor is not None:
            if torch.isnan(tensor).any() or torch.isinf(tensor).any():
                tqdm.write(f"  ❌  NaN/Inf in {attr_name} after densification — skipping this step")
                # Rollback
                try:
                    ckpt = torch.load(pre_ckpt, map_location=device, weights_only=False)
                    model.load_state_dict(ckpt["model_state"])
                    trainer._setup_optimizer()
                except Exception:
                    pass
                return n_before, False

    return n_after, True

# ── 8f. Metrics log init ──────────────────────────────────────────────────────
metrics_log   = []    # in-memory log, flushed to disk periodically
METRICS_PATH  = os.path.join(LOG_DIR, "metrics.json")
LOG_TXT_PATH  = os.path.join(LOG_DIR, "training_log.txt")

# ── 8g. Training state ────────────────────────────────────────────────────────
n           = len(model)
grad_accum  = torch.zeros(n, device=device)
grad_denom  = torch.zeros(n, device=device)
nan_count   = 0
oom_count   = 0
loss_val    = 0.0

# Abort if consecutive NaN is too high (indicates model divergence)
MAX_CONSECUTIVE_NAN = 50
consecutive_nan     = 0

t_start = time.time()
_pbar_vram = {"allocated_gb": 0.0}  # updated every 50 iters


print("═" * 60)
print(f"  MonoSplat — VRAM-Aware Training")
print(f"  Job     : {JOB_ID}")
print(f"  GPU     : {torch.cuda.get_device_name(0)}")
print(f"  Iters   : {START_ITER:,} → {ITERATIONS:,}")
print(f"  Max G   : {MAX_GAUSSIANS:,}")
print(f"  Res     : {W}×{H}")
print(f"  gsplat  : {'yes' if GSPLAT_AVAILABLE else 'no (slow fallback)'}")
print("═" * 60)

if START_ITER >= ITERATIONS:
    print("✅  Training already complete (checkpoint at final iteration).")
    print("   Run Cell 9 for previews or Cell 11 to export.")
else:
    # ── 8h. Main training loop ────────────────────────────────────────────────
    pbar = tqdm(range(START_ITER, ITERATIONS), desc="Training", dynamic_ncols=True)

    for it in pbar:

        # ── Randomly sample a training image ─────────────────────────────────
        idx    = np.random.randint(len(train_cameras))
        camera = train_cameras[idx]
        gt_img = train_images[idx].to(device, non_blocking=True)

        try:
            # ── Forward pass ──────────────────────────────────────────────────
            optimizer.zero_grad(set_to_none=True)
            rendered = renderer.render_torch(model, camera)
            loss     = combined_loss(rendered, gt_img, lambda_ssim=lambda_dssim)

            # ── NaN detection ─────────────────────────────────────────────────
            if torch.isnan(loss):
                nan_count       += 1
                consecutive_nan += 1
                if consecutive_nan <= 5 or consecutive_nan % 25 == 0:
                    tqdm.write(f"[{it:5d}] ⚠️  NaN loss (total {nan_count}, consecutive {consecutive_nan})")
                if consecutive_nan >= MAX_CONSECUTIVE_NAN:
                    tqdm.write(f"[{it:5d}] ❌  Too many consecutive NaN losses — saving emergency checkpoint")
                    save_checkpoint(it, loss_val, label="emergency_nan")
                    break
                continue
            else:
                consecutive_nan = 0

            # ── Backward pass ─────────────────────────────────────────────────
            loss.backward()
            # Gradient clipping prevents exploding gradients that cause NaN
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # ── Accumulate position gradients for densification ───────────────
            if model._positions.grad is not None:
                with torch.no_grad():
                    gn = model._positions.grad.norm(dim=1)
                    if gn.shape[0] == grad_accum.shape[0]:
                        grad_accum += gn
                        grad_denom += 1.0

            optimizer.step()
            trainer._apply_position_lr_decay(it, ITERATIONS)
            loss_val = loss.item()

        except torch.cuda.OutOfMemoryError:
            # ── OOM handling: emergency prune + continue ──────────────────────
            oom_count += 1
            torch.cuda.empty_cache(); gc.collect()
            tqdm.write(f"[{it:5d}] ⚠️  CUDA OOM #{oom_count} — emergency pruning to 60% cap")

            emergency_cap = int(MAX_GAUSSIANS * 0.6)
            cur_n = len(model)
            if cur_n > emergency_cap:
                n_prune = cur_n - emergency_cap
                with torch.no_grad():
                    _, low_idx = model.opacities.squeeze().topk(n_prune, largest=False)
                    mask = torch.zeros(cur_n, dtype=torch.bool, device=device)
                    mask[low_idx] = True
                model.prune_points(mask)
                trainer._setup_optimizer()
                optimizer  = trainer.optimizer
                n          = len(model)
                grad_accum = torch.zeros(n, device=device)
                grad_denom = torch.zeros(n, device=device)
                tqdm.write(f"       Pruned to {n:,} Gaussians")

            # Save emergency checkpoint after OOM
            if oom_count <= 3:
                save_checkpoint(it, loss_val, label="oom")

            if oom_count >= 5:
                tqdm.write(f"[{it:5d}] ❌  5+ OOM events — aborting. Lower MAX_GAUSSIANS.")
                break
            continue

        except Exception as e:
            tqdm.write(f"[{it:5d}] ❌  Unexpected error: {e}")
            save_checkpoint(it, loss_val, label="crash")
            raise

        # ── SH degree schedule ────────────────────────────────────────────────
        # Gradually increase SH complexity for more view-dependent appearance.
        # Doing this all at once at init causes instability.
        if it > 0 and it % 1000 == 0:
            model.oneup_sh_degree()

        # ── VRAM monitoring ───────────────────────────────────────────────────
        if it % 100 == 0:
            vs = vram_stats()
            if vs["allocated_gb"] > VRAM_WARN_GB:
                tqdm.write(f"[{it:5d}] ⚠️  VRAM: {vs['allocated_gb']:.1f} GB allocated (warn at {VRAM_WARN_GB})")

            # Emergency VRAM protection
            if vs["allocated_gb"] > VRAM_ABORT_GB:
                tqdm.write(f"[{it:5d}] 🚨  VRAM critical ({vs['allocated_gb']:.1f} GB) — emergency prune")
                emergency_cap = int(len(model) * 0.7)
                n_prune = len(model) - emergency_cap
                if n_prune > 0:
                    with torch.no_grad():
                        _, low_idx = model.opacities.squeeze().topk(n_prune, largest=False)
                        mask = torch.zeros(len(model), dtype=torch.bool, device=device)
                        mask[low_idx] = True
                    model.prune_points(mask)
                    trainer._setup_optimizer()
                    optimizer  = trainer.optimizer
                    n          = len(model)
                    grad_accum = torch.zeros(n, device=device)
                    grad_denom = torch.zeros(n, device=device)
                    torch.cuda.empty_cache(); gc.collect()

        # ── Densification ─────────────────────────────────────────────────────
        if DENSIFY_FROM <= it <= DENSIFY_UNTIL and it % DENSIFY_INTERVAL == 0:
            n_before = len(model)
            n_new, ok = safe_densify(grad_accum, grad_denom, it)

            # Rebuild optimizer and gradient buffers after densification
            optimizer  = trainer.optimizer
            n          = len(model)
            grad_accum = torch.zeros(n, device=device)
            grad_denom = torch.zeros(n, device=device)

            if ok:
                tqdm.write(f"[{it:5d}] 🔷 Densified: {n_before:,} → {n_new:,} Gaussians")
            else:
                tqdm.write(f"[{it:5d}] ℹ️  Densification skipped (cap={MAX_GAUSSIANS:,} or failed)")

        # ── Checkpoint saving ─────────────────────────────────────────────────
        if it > 0 and it % SAVE_INTERVAL == 0:
            ckpt_path = save_checkpoint(it, loss_val)
            tqdm.write(f"[{it:5d}] 💾 Checkpoint saved: {os.path.basename(ckpt_path)}")

        # ── Periodic VRAM clear ────────────────────────────────────────────────
        if it % 500 == 0:
            torch.cuda.empty_cache()

        # ── Metrics logging ───────────────────────────────────────────────────
        if it % 50 == 0:
            vs    = vram_stats()
            entry = {
                "iteration":     it,
                "loss":          round(loss_val, 6),
                "n_gaussians":   len(model),
                "vram_alloc_gb": round(vs.get("allocated_gb", 0.0), 2),
                "vram_resv_gb":  round(vs.get("reserved_gb",  0.0), 2),
                "vram_peak_gb":  round(vs.get("peak_gb",       0.0), 2),
                "nan_count":     nan_count,
                "oom_count":     oom_count,
                "elapsed_s":     round(time.time() - t_start, 1),
            }
            metrics_log.append(entry)

            # Flush metrics to disk every 500 iters — atomic write
            if it % 500 == 0:
                try:
                    _tmp = METRICS_PATH + ".tmp"
                    with open(_tmp, "w") as f:
                        json.dump(metrics_log, f, indent=2)
                    import os; os.replace(_tmp, METRICS_PATH)
                    sync_to_drive(METRICS_PATH, "logs")
                except Exception:
                    pass

        # ── Progress bar ──────────────────────────────────────────────────────
        # VRAM sampled every 50 iters to avoid synchronize() overhead every step
        if it % 50 == 0:
            _pbar_vram = vram_stats()
        pbar.set_postfix(
            loss=f"{loss_val:.4f}",
            N=f"{len(model):,}",
            VRAM=f"{_pbar_vram.get('allocated_gb', 0.0):.2f}G",
            NaN=nan_count,
            OOM=oom_count,
        )

    # ── Final checkpoint ──────────────────────────────────────────────────────
    save_checkpoint(ITERATIONS, loss_val, label="final")

    elapsed = time.time() - t_start

    # ── Record final iteration in log before flush ────────────────────────────
    # This guarantees the log's last entry matches the trainer's loss_val,
    # preventing the metrics summary from showing a stale mid-training value.
    _final_vs = vram_stats()
    metrics_log.append({
        "iteration":     ITERATIONS,
        "loss":          round(loss_val, 6),
        "n_gaussians":   len(model),
        "vram_alloc_gb": round(_final_vs.get("allocated_gb", 0.0), 2),
        "vram_resv_gb":  round(_final_vs.get("reserved_gb",  0.0), 2),
        "vram_peak_gb":  round(_final_vs.get("peak_gb",       0.0), 2),
        "nan_count":     nan_count,
        "oom_count":     oom_count,
        "elapsed_s":     round(elapsed, 1),
        "_is_final":     True,
    })
    try:
        _tmp = METRICS_PATH + ".tmp"
        with open(_tmp, "w") as f:
            json.dump(metrics_log, f, indent=2)
        import os; os.replace(_tmp, METRICS_PATH)
    except Exception as _e:
        print(f"⚠️  Final metrics flush failed: {_e}")

    # ── Validate: log final_loss must match trainer final_loss ─────────────────
    _log_last = metrics_log[-1]["loss"] if metrics_log else None
    if _log_last is not None and abs(_log_last - loss_val) > 1e-4:
        print(f"⚠️  Metrics consistency warning:")
        print(f"    Trainer final_loss  = {loss_val:.6f}")
        print(f"    Log last entry loss = {_log_last:.6f}")
        print(f"    Difference          = {abs(_log_last - loss_val):.6f}")
        print("    The metrics summary may show a stale value from an earlier flush.")
    else:
        print(f"✅  Metrics consistency check passed (final_loss = {loss_val:.4f})")

    print(f"\n{'═'*60}")
    print(f"  TRAINING COMPLETE")
    print(f"{'═'*60}")
    print(f"  Gaussians   : {len(model):,}")
    print(f"  Final loss  : {loss_val:.4f}")
    print(f"  NaN skipped : {nan_count}")
    print(f"  OOM events  : {oom_count}")
    print(f"  Peak VRAM   : {_final_vs.get('peak_gb', 0.0):.2f} GB")
    print(f"  Elapsed     : {elapsed/60:.1f} min")
    print(f"  Output dir  : {OUTPUT_DIR}")
    print(f"{'═'*60}")
    print("\n→  Run Cell 9 to render previews.")


---
## Cell 9 — Preview Rendering

Renders views from training cameras and saves them as preview images.  
Synced to Drive automatically. Use this to verify quality before export.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 9 — PREVIEW RENDERING
#  Purpose : Render preview images from training cameras.
#  Safety  : Each render is wrapped in try/except — a failed render does not
#            abort the preview run or corrupt the model.
# ══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import torch
import imageio
from PIL import Image

# Number of preview frames to render (evenly spaced through training cameras)
N_PREVIEWS = min(12, len(train_cameras))

print(f"Rendering {N_PREVIEWS} preview frames...")
print(f"Output: {PREVIEW_DIR}")

model.eval()  # disable training-only operations during rendering
indices = np.linspace(0, len(train_cameras) - 1, N_PREVIEWS, dtype=int)

rendered_paths = []
failed = 0

with torch.no_grad():
    for i, cam_idx in enumerate(indices):
        camera = train_cameras[cam_idx]
        gt_img = train_images[cam_idx]

        try:
            rendered = renderer.render_torch(model, camera)

            # Convert rendered tensor → PIL image
            render_np = rendered.cpu().permute(1, 2, 0).numpy()
            render_np = np.clip(render_np, 0, 1)
            render_pil = Image.fromarray((render_np * 255).astype(np.uint8))

            # Also load GT for side-by-side comparison
            gt_np  = gt_img.permute(1, 2, 0).numpy()
            gt_pil = Image.fromarray((gt_np * 255).astype(np.uint8))

            # Stitch side by side
            combined = Image.new("RGB", (W * 2, H))
            combined.paste(gt_pil,     (0, 0))
            combined.paste(render_pil, (W, 0))

            out_path = os.path.join(PREVIEW_DIR, f"preview_{i:03d}_cam{cam_idx:04d}.png")
            combined.save(out_path)
            rendered_paths.append(out_path)
            sync_to_drive(out_path, "previews")
            print(f"  ✅  preview_{i:03d}  (cam {cam_idx})")

        except Exception as e:
            failed += 1
            print(f"  ⚠️  Render failed for cam {cam_idx}: {e}")

model.train()

print(f"\n✅  {len(rendered_paths)} previews saved to: {PREVIEW_DIR}")
if failed:
    print(f"⚠️  {failed} renders failed (see above)")

# ── Display a sample in the notebook ─────────────────────────────────────────
if rendered_paths:
    from IPython.display import display
    sample = Image.open(rendered_paths[len(rendered_paths)//2])
    # Downscale for display if too large
    display_w = min(sample.width, 900)
    scale     = display_w / sample.width
    display_h = int(sample.height * scale)
    display(sample.resize((display_w, display_h), Image.LANCZOS))
    print("(Left: Ground Truth | Right: Gaussian Splat Render)")

print("\n→  Run Cell 10 to view training metrics.")


---
## Cell 10 — Metrics & Logging

Displays training metrics: loss curve, Gaussian count, VRAM usage.  
All metrics are saved to `logs/metrics.json` and synced to Drive.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 10 — METRICS & LOGGING
#  Purpose : Display training metrics and write structured log files.
# ══════════════════════════════════════════════════════════════════════════════

import json, os, math

# ── 10a. Load metrics ─────────────────────────────────────────────────────────
# Always reload from disk to ensure the display reflects the most recent flush.
# This prevents the stale-metric divergence where in-memory metrics_log differs
# from what was actually written.
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH) as f:
        metrics_log = json.load(f)
    print(f"✅  Loaded {len(metrics_log)} metric entries from {METRICS_PATH}")
    # Detect stale flushes: the last entry's loss should be the true final loss.
    # If the log was last flushed mid-training, the final entry won't be the
    # actual last value — the training summary will have caught this already.
    _log_final_loss = metrics_log[-1]["loss"] if metrics_log else None
    if _log_final_loss is not None:
        print(f"   Log final entry: iter={metrics_log[-1]['iteration']}  loss={_log_final_loss:.4f}")
elif not metrics_log:
    print("⚠️  No metrics recorded yet. Run Cell 8 first.")

# ── 10b. Summary table ────────────────────────────────────────────────────────
if metrics_log:
    iters  = [e["iteration"]     for e in metrics_log]
    losses = [e["loss"]          for e in metrics_log]
    n_gauss = [e["n_gaussians"]  for e in metrics_log]
    vrams  = [e["vram_alloc_gb"] for e in metrics_log]

    first, last = metrics_log[0], metrics_log[-1]
    mid   = metrics_log[len(metrics_log)//2]

    print("\n" + "═" * 55)
    print("  TRAINING METRICS SUMMARY")
    print("═" * 55)
    print(f"  Iterations logged   : {len(metrics_log)}")
    print(f"  Iter range          : {first['iteration']} → {last['iteration']}")
    print(f"  Loss (start)        : {first['loss']:.4f}")
    print(f"  Loss (mid)          : {mid['loss']:.4f}")
    print(f"  Loss (final)        : {last['loss']:.4f}")
    print(f"  Loss improvement    : {(1 - last['loss']/max(first['loss'], 1e-8))*100:.1f}%")
    print(f"  Gaussians (start)   : {first['n_gaussians']:,}")
    print(f"  Gaussians (final)   : {last['n_gaussians']:,}")
    # Use vram_peak_gb field if available (added in hardening pass)
    _peak_vrams = [e.get("vram_peak_gb", e.get("vram_alloc_gb", 0.0)) for e in metrics_log]
    print(f"  Peak VRAM           : {max(_peak_vrams):.2f} GB")
    print(f"  Final VRAM          : {last['vram_alloc_gb']:.2f} GB")
    print(f"  Total NaN events    : {last['nan_count']}")
    print(f"  Total OOM events    : {last['oom_count']}")
    print(f"  Elapsed             : {last['elapsed_s']/60:.1f} min")
    print("═" * 55)

    # ── 10c. ASCII loss curve ─────────────────────────────────────────────────
    # Simple ASCII plot — no matplotlib dependency required.
    print("\n  Loss curve (sampled):")
    n_buckets = 40
    bucket_size = max(1, len(losses) // n_buckets)
    buckets = [losses[i*bucket_size] for i in range(min(n_buckets, len(losses)))]

    if buckets:
        lo, hi = min(buckets), max(buckets)
        r = hi - lo if hi > lo else 1
        HEIGHT = 8
        chart = []
        for row in range(HEIGHT, 0, -1):
            threshold = lo + (row / HEIGHT) * r
            line = ""
            for val in buckets:
                line += "█" if val >= threshold else " "
            y_label = f"{lo + (row/HEIGHT)*r:.4f}"
            chart.append(f"  {y_label} │{line}│")
        for row in chart:
            print(row)
        print(f"  {'─'*(len(buckets)+2+8)}")
        print(f"  start{' '*(len(buckets)-7)}end")

# ── 10d. Write structured log ─────────────────────────────────────────────────
if metrics_log:
    with open(METRICS_PATH, "w") as f:
        json.dump(metrics_log, f, indent=2)

    # Human-readable training log
    with open(LOG_TXT_PATH, "w") as f:
        f.write(f"MonoSplat Training Log\n")
        f.write(f"Job ID: {JOB_ID}\n")
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
        f.write(f"Resolution: {W}×{H}\n")
        f.write(f"Max Gaussians: {MAX_GAUSSIANS:,}\n")
        f.write(f"\n{'iter':>8}  {'loss':>10}  {'gaussians':>12}  {'vram_gb':>8}\n")
        f.write("-" * 50 + "\n")
        for e in metrics_log:
            f.write(f"{e['iteration']:>8}  {e['loss']:>10.4f}  {e['n_gaussians']:>12,}  {e['vram_alloc_gb']:>8.2f}\n")

    sync_to_drive(METRICS_PATH,  "logs")
    sync_to_drive(LOG_TXT_PATH,  "logs")
    print(f"\n✅  Metrics saved: {METRICS_PATH}")
    print(f"✅  Log saved:     {LOG_TXT_PATH}")

print("\n→  Run Cell 11 to export .splat and .ply files.")


---
## Cell 11 — Export

Exports the final Gaussian splat in all formats, packages everything into
`final_export.zip`, and downloads to your computer.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 11 — EXPORT
#  Purpose : Convert .ply → .splat, bundle everything, download.
# ══════════════════════════════════════════════════════════════════════════════

import os, glob, shutil, zipfile
from pathlib import Path

# ── 11a. Find best checkpoint .ply ────────────────────────────────────────────
ply_files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "*.ply")),
    key=os.path.getmtime
)

if not ply_files:
    print("❌  No .ply files found in output directory.")
    print(f"   Expected: {OUTPUT_DIR}")
    print("   Make sure Cell 8 completed and saved a checkpoint.")
    raise SystemExit("No output files.")

latest_ply = ply_files[-1]
print(f"📄  Latest .ply checkpoint: {latest_ply}")
print(f"    Size: {os.path.getsize(latest_ply)/1e6:.1f} MB")

# ── 11b. Convert .ply → .splat ────────────────────────────────────────────────
splat_path = os.path.join(OUTPUT_DIR, f"{JOB_ID}.splat")
ply_dest   = os.path.join(OUTPUT_DIR, f"{JOB_ID}.ply")

try:
    from src.utils.io_utils import save_splat, load_ply
    print("\n🔄  Converting .ply → .splat ...")
    gaussians = load_ply(latest_ply)
    save_splat(splat_path, gaussians)
    print(f"✅  .splat: {splat_path}  ({os.path.getsize(splat_path)/1e6:.1f} MB)")
except Exception as e:
    print(f"⚠️  .splat conversion failed: {e}")
    print("   The .ply file is still usable — drag to SuperSplat or Luma.")
    splat_path = None

# Copy .ply with job_id name
if os.path.abspath(latest_ply) != os.path.abspath(ply_dest):
    shutil.copy2(latest_ply, ply_dest)
print(f"✅  .ply:   {ply_dest}  ({os.path.getsize(ply_dest)/1e6:.1f} MB)")

# ── 11c. Bundle export zip ────────────────────────────────────────────────────
EXPORT_ZIP = os.path.join(OUTPUT_DIR, f"{JOB_ID}_final_export.zip")
files_to_bundle = []

if splat_path and os.path.exists(splat_path):
    files_to_bundle.append((splat_path,   f"{JOB_ID}.splat"))
files_to_bundle.append((ply_dest,         f"{JOB_ID}.ply"))
if os.path.exists(METRICS_PATH):
    files_to_bundle.append((METRICS_PATH, "metrics.json"))
if os.path.exists(LOG_TXT_PATH):
    files_to_bundle.append((LOG_TXT_PATH, "training_log.txt"))

print(f"\n📦  Bundling export zip: {EXPORT_ZIP}")
with zipfile.ZipFile(EXPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in files_to_bundle:
        if os.path.exists(src):
            zf.write(src, arcname)
            print(f"    + {arcname}")

print(f"\n✅  Export bundle: {EXPORT_ZIP}  ({os.path.getsize(EXPORT_ZIP)/1e6:.1f} MB)")

# Sync to Drive
sync_to_drive(EXPORT_ZIP, "exports")
if splat_path and os.path.exists(splat_path):
    sync_to_drive(splat_path, "exports")
sync_to_drive(ply_dest, "exports")
print("✅  Files synced to Drive/MonoSplat/exports/")

# ── 11d. Download ─────────────────────────────────────────────────────────────
from google.colab import files

print("\n⬇️   Downloading files...")

if splat_path and os.path.exists(splat_path):
    print(f"    Downloading {JOB_ID}.splat ...")
    files.download(splat_path)

print(f"    Downloading {JOB_ID}.ply ...")
files.download(ply_dest)

# ── 11e. Next-step instructions ───────────────────────────────────────────────
print(f"\n{'═'*60}")
print("  NEXT STEPS — on your desktop")
print("═" * 60)
print(f"  1. Create folder:  work/{JOB_ID}/models/gaussian/")
print(f"  2. Move files there:")
print(f"       {JOB_ID}.splat")
print(f"       {JOB_ID}.ply")
print(f"  3. Update models/registry.json:")
print(f'       "status":     "ready",')
print(f'       "splat_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.splat",')
print(f'       "ply_path":   "work/{JOB_ID}/models/gaussian/{JOB_ID}.ply"')
print(f"  4. Open: http://localhost:8000/viewer/{JOB_ID}")
print(f"  Or drag .splat to: https://supersplat.playcanvas.com")
print("═" * 60)


---
## Cell 12 — Debug Utilities

Use this cell when something goes wrong. It provides tools to inspect the
model state, VRAM usage, checkpoint contents, and diagnose common failure modes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 12 — DEBUG UTILITIES
#  Purpose : Diagnose failures. Run any section independently.
# ══════════════════════════════════════════════════════════════════════════════

import os, glob, json
import torch

print("═" * 55)
print("  DEBUG DIAGNOSTICS")
print("═" * 55)

# ── D1. VRAM status ───────────────────────────────────────────────────────────
print("\n[D1] VRAM Status:")
alloc = torch.cuda.memory_allocated() / 1e9
resrv = torch.cuda.memory_reserved()  / 1e9
free  = torch.cuda.mem_get_info()[0]  / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  Allocated : {alloc:.3f} GB")
print(f"  Reserved  : {resrv:.3f} GB")
print(f"  Free      : {free:.3f} GB")
print(f"  Total     : {total:.3f} GB")
print(f"  Usage     : {alloc/total*100:.1f}%")

# ── D2. Model state ────────────────────────────────────────────────────────────
print("\n[D2] Gaussian Model State:")
try:
    n = len(model)
    print(f"  Count     : {n:,}")
    for attr_name in ["_positions", "_scales", "_rotations", "_opacities", "_features_dc"]:
        t = getattr(model, attr_name, None)
        if t is not None:
            has_nan = torch.isnan(t).any().item()
            has_inf = torch.isinf(t).any().item()
            flag = " ❌NaN" if has_nan else (" ❌Inf" if has_inf else "")
            print(f"  {attr_name:<20}  shape={tuple(t.shape)}  min={t.min().item():.4f}  max={t.max().item():.4f}{flag}")
except NameError:
    print("  ⚠️  model not defined yet (run Cell 7 first)")

# ── D3. Checkpoint inventory ───────────────────────────────────────────────────
print("\n[D3] Checkpoint Inventory:")
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*.ckpt")))
if ckpts:
    for ckpt in ckpts[-5:]:  # show last 5
        size = os.path.getsize(ckpt) / 1e6
        print(f"  {os.path.basename(ckpt):<40}  {size:.1f} MB")
    if len(ckpts) > 5:
        print(f"  ... ({len(ckpts)-5} older checkpoints not shown)")
else:
    print("  ⚠️  No checkpoints found")

# ── D4. Common failure guide ───────────────────────────────────────────────────
print("\n[D4] Common Failures:")
print("""
  SYMPTOM: Kernel crashes immediately after training starts
  CAUSE  : VRAM exhausted before first checkpoint
  FIX    : Lower MAX_GAUSSIANS (try 50_000), lower resolution (480×270)

  SYMPTOM: NaN loss from iteration 0
  CAUSE  : Bad point cloud (extreme outliers) or config mismatch
  FIX    : Check Cell 6 validation warnings; try SH_DEGREE=0

  SYMPTOM: Loss not decreasing after 500 iters
  CAUSE  : Too few frames, bad COLMAP, camera registration failure
  FIX    : Check registration ratio in Cell 6; re-run COLMAP on desktop

  SYMPTOM: OOM after densification
  CAUSE  : Densification doubled Gaussian count past VRAM capacity
  FIX    : Lower MAX_GAUSSIANS by 30%; increase DENSIFY_INTERVAL

  SYMPTOM: Training very slow (>10s/iter)
  CAUSE  : gsplat not installed — using Python fallback renderer
  FIX    : Re-run Cell 2; check GSPLAT_AVAILABLE flag
""")

# ── D5. Quick checkpoint load test ────────────────────────────────────────────
print("\n[D5] Checkpoint integrity test:")
if ckpts:
    latest = ckpts[-1]
    try:
        ck = torch.load(latest, map_location="cpu", weights_only=False)
        print(f"  ✅  {os.path.basename(latest)}")
        print(f"      iteration   : {ck.get('iteration', 'N/A')}")
        print(f"      loss        : {ck.get('loss', 'N/A')}")
        print(f"      n_gaussians : {ck.get('n_gaussians', 'N/A')}")
        print(f"      has config  : {'config' in ck}")
    except Exception as e:
        print(f"  ❌  Failed to load {os.path.basename(latest)}: {e}")
else:
    print("  ⚠️  No checkpoint to test")

print("\n═" * 28)


---
## Cell 13 — Benchmark Mode

Run a short benchmark to validate your environment before committing to a full training run.  
Tests 100 iterations on a synthetic point cloud and reports per-iteration timing and VRAM usage.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 13 — BENCHMARK MODE
#  Purpose : Validate environment stability with a short synthetic benchmark.
#  When to run: BEFORE your first real training run, to verify:
#    - gsplat is working
#    - VRAM is stable across 100 iterations
#    - No immediate OOM or NaN
#    - Estimate per-iteration time for your GPU
# ══════════════════════════════════════════════════════════════════════════════

import time, gc
import numpy as np
import torch

print("═" * 55)
print("  BENCHMARK MODE — 100-iteration synthetic test")
print("═" * 55)
print(f"  GPU     : {torch.cuda.get_device_name(0)}")
print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  gsplat  : {'yes' if GSPLAT_AVAILABLE else 'no (slow fallback)'}")
print()

# ── Create a minimal synthetic Gaussian model ─────────────────────────────────
N_SYNTH = 5000   # small enough to be fast
np.random.seed(0)
synth_xyz = np.random.randn(N_SYNTH, 3).astype(np.float32) * 2.0
synth_rgb = np.random.rand(N_SYNTH, 3).astype(np.float32)

bench_model = GaussianModel(sh_degree=1)
bench_model = bench_model.to(device)
bench_model.create_from_points(synth_xyz, synth_rgb)

bench_renderer = GaussianRenderer(
    width=640, height=360,
    bg_color=[0, 0, 0],
    device=device,
    batch_size=1,
    use_gsplat=GSPLAT_AVAILABLE,
)

# Use first available training camera
bench_camera = train_cameras[0]
bench_gt     = train_images[0].to(device)
bench_opt    = torch.optim.Adam(bench_model.parameters(), lr=1e-3)

N_BENCH_ITERS = 100
timings = []
vrams   = []
nan_cnt = 0

print(f"Running {N_BENCH_ITERS} iterations...")
torch.cuda.empty_cache(); gc.collect()

for i in range(N_BENCH_ITERS):
    t0 = time.time()
    try:
        bench_opt.zero_grad(set_to_none=True)
        rendered = bench_renderer.render_torch(bench_model, bench_camera)
        loss     = combined_loss(rendered, bench_gt)
        if torch.isnan(loss):
            nan_cnt += 1
            continue
        loss.backward()
        bench_opt.step()
    except torch.cuda.OutOfMemoryError:
        print(f"  ❌  OOM at iteration {i} — this GPU may be too constrained")
        break

    t1 = time.time()
    timings.append(t1 - t0)
    vrams.append(torch.cuda.memory_allocated() / 1e9)

# ── Cleanup bench model ───────────────────────────────────────────────────────
del bench_model, bench_renderer, bench_opt
torch.cuda.empty_cache(); gc.collect()

# ── Results ───────────────────────────────────────────────────────────────────
if timings:
    avg_ms   = np.mean(timings) * 1000
    p95_ms   = np.percentile(timings, 95) * 1000
    est_min  = (avg_ms / 1000) * ITERATIONS / 60

    print(f"\n  Results:")
    print(f"  Per-iteration (avg) : {avg_ms:.0f} ms")
    print(f"  Per-iteration (p95) : {p95_ms:.0f} ms")
    print(f"  NaN losses          : {nan_cnt}")
    print(f"  Peak VRAM           : {max(vrams):.3f} GB")
    print(f"\n  Estimated training time for {ITERATIONS:,} iters: ~{est_min:.0f} min")

    if nan_cnt > 0:
        print(f"\n  ⚠️  NaN losses occurred ({nan_cnt}) — check your renderer config")
    else:
        print(f"\n  ✅  No NaN losses — renderer is numerically stable")

    if not GSPLAT_AVAILABLE:
        print(f"\n  ⚠️  gsplat not available — actual training will be ~{int(avg_ms/50)}× slower")
        print(f"       Consider re-running Cell 2 to install gsplat.")

print("\n✅  Benchmark complete. Ready for full training.")
